In [0]:
# Celda 1: instalar la librería para descargar desde Kaggle
%pip install kagglehub
# EJECUTAR SOLO UNA VEZ: descarga inicial

In [0]:
# Celda 2: crear un "volumen", el espacio de almacenamiento de archivos en Databricks
# (equivalente a una librería permanente en SAS)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw")
# EJECUTAR SOLO UNA VEZ: descarga inicial

In [0]:
# EJECUTAR SOLO UNA VEZ: descarga inicial
# # Celda 3: descargar el dataset y copiarlo al volumen
import kagglehub, shutil, os

path = kagglehub.dataset_download("wordsforthewise/lending-club")  # descarga a disco temporal
print("Descargado en:", path)

destino = "/Volumes/workspace/default/raw/lending_club"
shutil.copytree(path, destino, dirs_exist_ok=True)  # copia al volumen permanente

for root, dirs, files in os.walk(destino):  # lista los archivos y su tamaño
    for f in files:
        ruta = os.path.join(root, f)
        print(f, round(os.path.getsize(ruta) / 1e6, 1), "MB")


In [0]:
# Celda 4: ubicar las rutas exactas de los CSV dentro del volumen
import glob
archivos = glob.glob("/Volumes/workspace/default/raw/lending_club/**/*.csv", recursive=True)
for a in archivos:
    print(a)


In [0]:
# Celda 5: función para leer un CSV y guardarlo como tabla
import re
import os

def csv_a_tabla(ruta, nombre_tabla):
    df = (spark.read
          .option("header", True)       # primera fila = nombres de columnas
          .option("inferSchema", True)  # Spark deduce el tipo de cada columna (número, texto…)
          .option("multiLine", True)    # hay textos libres con saltos de línea
          .option("escape", '"')
          .csv(ruta))
    # Las tablas no aceptan espacios ni símbolos en los nombres de columna:
    # "Amount Requested" -> "amount_requested"
    for c in df.columns:
        df = df.withColumnRenamed(c, re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_"))
    df.write.mode("overwrite").saveAsTable(nombre_tabla)  # como un DATA lib.tabla en SAS
    t = spark.table(nombre_tabla)
    print(f"{nombre_tabla}: {t.count():,} filas, {len(t.columns)} columnas")


ruta_acc = [a for a in archivos if "accepted" in a and os.path.isfile(a)][0]
ruta_rej = [a for a in archivos if "rejected" in a and os.path.isfile(a)][0]

csv_a_tabla(ruta_acc, "workspace.default.lc_accepted_raw")
csv_a_tabla(ruta_rej, "workspace.default.lc_rejected_raw")
